# Correlation analysis of markers and animals.

It uses the file generated by `fibers_length_multi.ipynb` from Cuisto (https://teamncmc.github.io/cuisto/).

What this notebook does:

1. The overall correlation between markers (using only `hemisphere = both`).
2. A PCA of regions based on markers (using `hemisphere = both`).
3. For each marker, an animal x animal correlation heatmap based on regional profiles.

How to use the notebook:

1. Replace with the full path to your `df_regions.xlsx` file.
2. Replace with the full path to the directory where figures will be saved.
3. Run All

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ------------------------------------------------------------------
# full path to the df_regions.xlsx file (1)
excel_path = r"D:\path\to\df_regions.xlsx"
# path to the save directory (2)
output_dir = Path(r"D:\path\to\save\directory")
output_dir.mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------

excel_path = Path(excel_path)
if not excel_path.exists():
    raise FileNotFoundError(f"File not found : {excel_path}") # Check if the Excel file exists

# Select the library to read the Excel file based on the extension
ext = excel_path.suffix.lower()
if ext in [".xlsx", ".xlsm", ".xltx", ".xltm"]:
    engine = "openpyxl"
elif ext == ".xls":
    engine = "xlrd"
else:
    engine = "openpyxl"  # Default

df = pd.read_excel(excel_path, engine=engine) # Excel file read as Pandas.DataFrame (df)

print("Columns: ")
print(df.columns.tolist())

In [ ]:
# Select density colonne
density_candidates = [c for c in df.columns if "density" in str(c).lower()]
if not density_candidates:
    raise ValueError("No density columns found.") # If column not found

density_col = density_candidates[0] # First density column found and used
print(f"Density column used: {density_col!r}")

channel_col = 'channel'  # column name for markers
region_col = 'Name'      # column name for regions
animal_col = 'animal'    # column name for animals
hemi_col = 'hemisphere'  # column name for hemisphere

# Check if all column names are found in df
for col in [channel_col, region_col, animal_col, hemi_col]:
    if col not in df.columns:
        raise KeyError(f"Column not found in folder: {col!r}")

## 1. Overall correlation between markers

This section: 
- Filter only the rows where "hemisphere == “both".
- Calculate a regions × markers correlation coefficients table based on average density.
- Display the overall correlation between markers.

In [ ]:
# --------------------------------------------------------------
# Keep only rows where hemisphere == "both"
# --------------------------------------------------------------
df["hemisphere_norm"] = df[hemi_col].astype(str).str.lower()
df_both = df[df["hemisphere_norm"] == "both"].copy()

print(f"Number of rows with hemisphere='both': {len(df_both)}")

# --------------------------------------------------------------
# Pivot df: rows = regions, columns = channels, values = mean density
# Rows with any NaN are dropped
# --------------------------------------------------------------
pivot_global = df_both.pivot_table(
    index=region_col,
    columns=channel_col,
    values=density_col,
    aggfunc="mean"
).dropna()

display(pivot_global.head())
print("Reshaped DataFrame shape: ", pivot_global.shape)

markers = pivot_global.columns
n_regions = pivot_global.shape[0]
print(f"Number of regions used: {n_regions}")

# --------------------------------------------------------------
# Heatmap: descriptive correlation between markers
# --------------------------------------------------------------
plt.figure(figsize=(6, 5))
sns.heatmap(
    pivot_global.corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Descriptive correlation between markers\n(hemisphere = both)")
plt.tight_layout()

plt.savefig(output_dir/"descriptive_correlation_between_markers.svg", format="svg")
plt.show()

## 2. PCA of regions by marker

This section:
- Filter regions
- Compute remotes regions and weighted average of the markers
- Calculate and display PCA of the regions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# --------------------------------------------------------------
# Regions filtering
# --------------------------------------------------------------

density_threshold = 0.0000000001
mean_density = pivot_global.mean(axis=1)
pivot_filtered = pivot_global[mean_density > density_threshold]

print("Number of regions used:", pivot_filtered.shape[0])

# --------------------------------------------------------------
# PCA
# --------------------------------------------------------------

X_scaled = StandardScaler().fit_transform(pivot_filtered.values) # Standardize X by removing the mean and scaling to unit variance

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_ * 100 # Percentage of variance explained by each of the selected components

# Scores contains the coordinates of each region in PCA space
scores = pd.DataFrame(
    X_pca,
    index=pivot_filtered.index,
    columns=["PC1", "PC2"]
)

# --------------------------------------------------------------
# Remotes regions
# --------------------------------------------------------------

top_PC1_pos = scores.sort_values("PC1", ascending=False).head(8)
top_PC1_neg = scores.sort_values("PC1", ascending=True).head(0)

top_PC2_pos = scores.sort_values("PC2", ascending=False).head(8)
top_PC2_neg = scores.sort_values("PC2", ascending=True).head(0)

extreme_regions = pd.concat([
    top_PC1_pos,
    top_PC1_neg,
    top_PC2_pos,
    top_PC2_neg
]).index.unique()

print("\nRemotes regions PC1+/PC1-: ")
display(pd.concat([top_PC1_pos, top_PC1_neg]))

print("\nRemotes regions PC2+/PC2-:")
display(pd.concat([top_PC2_pos, top_PC2_neg]))

# --------------------------------------------------------------
# Weighted average of the markers
# --------------------------------------------------------------

marker_centers = {}

for m in pivot_filtered.columns:
    weights = pivot_filtered[m].values.astype(float)
    if weights.sum() == 0:
        continue
    
    c1 = np.average(scores["PC1"], weights=weights)
    c2 = np.average(scores["PC2"], weights=weights)
    
    marker_centers[m] = (c1, c2)

marker_centers_df = pd.DataFrame(marker_centers, index=["PC1", "PC2"]).T

print("\nMarker centers: ")
display(marker_centers_df)

# --------------------------------------------------------------
# Plot and save results
# --------------------------------------------------------------

svg_path = output_dir/"PCA_regions_extremes_centers.svg"

fig, ax = plt.subplots(figsize=(9,8))

# All data points
ax.scatter(scores["PC1"], scores["PC2"], alpha=0.4)

# Remotes regions
ax.scatter(
    scores.loc[extreme_regions, "PC1"],
    scores.loc[extreme_regions, "PC2"],
    s=130
)

for region in extreme_regions:
    ax.text(
        scores.loc[region, "PC1"],
        scores.loc[region, "PC2"],
        region,
        fontsize=9,
        weight="bold"
    )

# Marker centers
for m, (c1, c2) in marker_centers.items():
    ax.scatter(c1, c2, s=300, marker="X")
    ax.text(c1, c2, f"  {m}", fontsize=13, weight="bold")

ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)")
ax.set_title("PCA of the remotes regions per axis\n + marker centers")
ax.grid(True)

fig.tight_layout()

# --------------------------------------------------------------
# Plot saving
# --------------------------------------------------------------
fig.savefig(svg_path, format="svg", bbox_inches="tight")

print("SVG saved ", svg_path)

plt.show()
plt.close(fig)

# --------------------------------------------------------------
# PCA region scores saving
# --------------------------------------------------------------
csv_path = output_dir/"PCA_region_scores.csv"

scores_out = scores.join(pivot_filtered)
scores_out["is_extreme"] = scores_out.index.isin(extreme_regions)

scores_out.to_csv(csv_path)

print("CSV saved: ", csv_path)

## 3. Region-level marker heterogeneity: coefficient of variation vs. mean density

This section:

- Computes the corrected CV per region by pivoting over channels and aggregating across animals
- Builds a region_stats dataframe combining mean marker density and CV
- Filters out low-density regions below a given threshold
- Plots CV vs. mean density to visualize marker heterogeneity across regions

In [ ]:
# --------------------------------------------------------------
# Per-region coefficient of variation(cv): pivot (region × animal) over channels, then
# compute std/mean per animal, and aggregate by median across animals
# --------------------------------------------------------------

# Pivot table: rows = (region, animal), columns = channels, values = mean density
pivot_region_animal = df_both.pivot_table(
    index=[region_col, animal_col],
    columns=channel_col,
    values=density_col,
    aggfunc="mean"
).dropna()

# Compute cv (std/mean) across channels for each (region, animal) pair
cv_region_animal = (
    pivot_region_animal.std(axis=1)/
    pivot_region_animal.mean(axis=1)
)

# Drop infinite or NaN values (e.g. mean == 0)
cv_region_animal = cv_region_animal.replace([np.inf, -np.inf], np.nan).dropna()

# Final cv per region: median across animals
cv_region = cv_region_animal.groupby(level=0).median()

print("cv computed :", cv_region.shape)

In [ ]:
# --------------------------------------------------------------
# Recreate region_stats using the corrected CV
# --------------------------------------------------------------

region_stats = pd.DataFrame({
    "mean_marker": pivot_global.mean(axis=1),
    "cv_marker": cv_region
}).dropna()

# --------------------------------------------------------------
# Filtering regions by average density
# --------------------------------------------------------------

density_threshold = 2e-5

region_stats_filt = region_stats[
    region_stats["mean_marker"] > density_threshold
]

print(f"Number of regions after filtering: {len(region_stats_filt)}")
print(f"Average density threshold: {density_threshold}")

# --------------------------------------------------------------
# Graph: CV vs average density
# --------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    region_stats_filt["mean_marker"],
    region_stats_filt["cv_marker"],
    alpha=0.7
)

# Mark all regions
for region in region_stats_filt.index:
    ax.text(
        region_stats_filt.loc[region, "mean_marker"],
        region_stats_filt.loc[region, "cv_marker"],
        region,
        fontsize=11
    )

ax.set_xlabel("Average marker density")
ax.set_ylabel("CV corrigé")
ax.set_title(
    "Regional variation in markers\n"
    f"(average density > {density_threshold:.1e})"
)

ax.grid(True)
fig.tight_layout()

fig.savefig(
    output_dir/"cv_vs_mean_density_regions_all_labels.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)